In [235]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, Dense, Conv2D, Flatten, Reshape, UpSampling2D, MaxPooling2D, concatenate
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from sklearn.metrics import balanced_accuracy_score,f1_score,precision_score,recall_score
from sklearn.metrics import confusion_matrix



In [236]:
## Load Gene Expression Data (CSV)
gene_data = pd.read_csv("concat1.csv") 


In [237]:
gene_data.head()

,sample,ARHGEF10L_x,HIF3A_x,RNF10_x,RNF11_x,RNF13_x,GTF2IP1_x,REM1_x,RTN4RL2_x,C16orf13_x,...,SELE,METTL11A,NPY5R,GNGT2,TULP3,PTRF,BCL6B,GSTK1,SELP,SELS
0,0,10.1570,3.8779,11.8626,10.6985,10.3806,12.2526,4.0582,9.3086,10.2299,...,8.2678,7.8548,2.4876,6.3782,8.6781,14.6956,9.7151,10.5910,9.5115,10.4914
1,1,11.0790,8.3144,12.0139,10.8516,10.0041,12.9142,4.9078,8.2973,10.8124,...,4.2475,8.6377,2.4018,7.2380,8.2495,13.6382,9.2774,11.2910,8.9992,11.0799
2,2,9.6560,4.0173,11.4924,11.0150,10.6400,12.3090,3.9681,8.8456,8.5291,...,9.5634,8.1028,4.3152,5.9360,8.8963,14.0056,10.5443,11.0894,9.9221,10.3887
3,3,10.1195,8.0548,11.5663,11.6184,10.3695,13.2329,4.0695,7.1342,9.7105,...,3.0112,7.7803,3.6205,5.6292,9.5608,14.1202,9.3961,10.7154,8.9942,9.9055
4,4,9.0858,9.4343,12.2037,11.0686,10.4942,11.8646,3.2241,7.7444,8.2831,...,9.6826,7.7932,4.3647,5.8571,8.8448,14.1496,10.2329,10.6978,10.5565,10.0059


In [238]:
gene_data.columns = gene_data.columns.str.strip()
print(gene_data.columns)

Index(['sample', 'ARHGEF10L_x', 'HIF3A_x', 'RNF10_x', 'RNF11_x', 'RNF13_x',
       'GTF2IP1_x', 'REM1_x', 'RTN4RL2_x', 'C16orf13_x',
       ...
       'SELE', 'METTL11A', 'NPY5R', 'GNGT2', 'TULP3', 'PTRF', 'BCL6B', 'GSTK1',
       'SELP', 'SELS'],
      dtype='object', length=46955)


In [239]:
print(gene_data.columns[0:10])

Index(['sample', 'ARHGEF10L_x', 'HIF3A_x', 'RNF10_x', 'RNF11_x', 'RNF13_x',
       'GTF2IP1_x', 'REM1_x', 'RTN4RL2_x', 'C16orf13_x'],
      dtype='object')


In [240]:
print(gene_data.columns[-1])
print(gene_data.shape)

SELS
(109, 46955)


In [241]:
missing_values = gene_data.isnull().sum()  # Get the count of missing values per column
print(missing_values)

sample          0
ARHGEF10L_x     0
HIF3A_x         0
RNF10_x         0
RNF11_x         0
               ..
PTRF           50
BCL6B          50
GSTK1          50
SELP           50
SELS           50
Length: 46955, dtype: int64


In [242]:
print(gene_data.head())

   sample  ARHGEF10L_x  HIF3A_x  RNF10_x  RNF11_x  RNF13_x  GTF2IP1_x  REM1_x  \
0       0      10.1570   3.8779  11.8626  10.6985  10.3806    12.2526  4.0582   
1       1      11.0790   8.3144  12.0139  10.8516  10.0041    12.9142  4.9078   
2       2       9.6560   4.0173  11.4924  11.0150  10.6400    12.3090  3.9681   
3       3      10.1195   8.0548  11.5663  11.6184  10.3695    13.2329  4.0695   
4       4       9.0858   9.4343  12.2037  11.0686  10.4942    11.8646  3.2241   

   RTN4RL2_x  C16orf13_x  ...    SELE  METTL11A   NPY5R   GNGT2   TULP3  \
0     9.3086     10.2299  ...  8.2678    7.8548  2.4876  6.3782  8.6781   
1     8.2973     10.8124  ...  4.2475    8.6377  2.4018  7.2380  8.2495   
2     8.8456      8.5291  ...  9.5634    8.1028  4.3152  5.9360  8.8963   
3     7.1342      9.7105  ...  3.0112    7.7803  3.6205  5.6292  9.5608   
4     7.7444      8.2831  ...  9.6826    7.7932  4.3647  5.8571  8.8448   

      PTRF    BCL6B    GSTK1     SELP     SELS  
0  14.6956   

In [243]:
column_means=gene_data.mean()
print(column_means)

sample         54.000000
ARHGEF10L_x     9.376374
HIF3A_x         6.919358
RNF10_x        11.806372
RNF11_x        10.831935
                 ...    
PTRF           13.948715
BCL6B           9.576671
GSTK1          10.929327
SELP            9.338775
SELS           10.345458
Length: 46955, dtype: float64


In [244]:
gene_data.fillna(gene_data.mean(), inplace=True)

In [245]:
print(gene_data.head())

   sample  ARHGEF10L_x  HIF3A_x  RNF10_x  RNF11_x  RNF13_x  GTF2IP1_x  REM1_x  \
0       0      10.1570   3.8779  11.8626  10.6985  10.3806    12.2526  4.0582   
1       1      11.0790   8.3144  12.0139  10.8516  10.0041    12.9142  4.9078   
2       2       9.6560   4.0173  11.4924  11.0150  10.6400    12.3090  3.9681   
3       3      10.1195   8.0548  11.5663  11.6184  10.3695    13.2329  4.0695   
4       4       9.0858   9.4343  12.2037  11.0686  10.4942    11.8646  3.2241   

   RTN4RL2_x  C16orf13_x  ...    SELE  METTL11A   NPY5R   GNGT2   TULP3  \
0     9.3086     10.2299  ...  8.2678    7.8548  2.4876  6.3782  8.6781   
1     8.2973     10.8124  ...  4.2475    8.6377  2.4018  7.2380  8.2495   
2     8.8456      8.5291  ...  9.5634    8.1028  4.3152  5.9360  8.8963   
3     7.1342      9.7105  ...  3.0112    7.7803  3.6205  5.6292  9.5608   
4     7.7444      8.2831  ...  9.6826    7.7932  4.3647  5.8571  8.8448   

      PTRF    BCL6B    GSTK1     SELP     SELS  
0  14.6956   

In [246]:
## feature value seperation
x=gene_data.iloc[:,0:-1]
y=gene_data.iloc[:,-1]
print(x.shape)
print(y.shape)

(109, 46954)
(109,)


In [247]:
### label encoding
label_encoder=LabelEncoder()
label_encoder.fit(y)
y_encoded=label_encoder.transform(y)
labels=label_encoder.classes_
classes=np.unique(y_encoded)

In [248]:
print(labels)
print(classes)

[ 9.9055      9.9683      9.9814      9.9983     10.0059     10.0244
 10.036      10.0597     10.06       10.0669     10.0754     10.0789
 10.0817     10.091      10.1235     10.1423     10.1448     10.1474
 10.1594     10.1668     10.1867     10.1891     10.1952     10.2014
 10.2187     10.2219     10.2284     10.2927     10.2977     10.3129
 10.3165     10.3209     10.3368     10.3396     10.34545763 10.3517
 10.3534     10.3574     10.3654     10.3825     10.3887     10.3939
 10.3999     10.4007     10.4583     10.4664     10.4914     10.4969
 10.5031     10.517      10.6213     10.6767     10.7805     10.8408
 10.9085     10.9355     11.0154     11.0799     11.1032     11.1174    ]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59]


In [249]:
##split
x_train,x_test,y_train,y_test = train_test_split(x,y_encoded,test_size=0.2,random_state=42)

In [250]:
gene_data.iloc[:,0:10].describe()

,sample,ARHGEF10L_x,HIF3A_x,RNF10_x,RNF11_x,RNF13_x,GTF2IP1_x,REM1_x,RTN4RL2_x,C16orf13_x
count,109.000000,109.000000,109.000000,109.000000,109.000000,109.000000,109.000000,109.000000,109.000000,109.000000
mean,54.000000,9.376374,6.919358,11.806372,10.831935,10.447471,12.265610,3.970443,6.601024,9.574548
std,31.609598,0.851307,2.337221,0.383198,0.431277,0.556061,0.533188,1.053793,1.858343,0.736150
min,0.000000,5.692200,1.210800,11.027200,9.702900,8.116800,10.801000,0.674500,1.702000,7.368200
25%,27.000000,9.025000,4.966000,11.548900,10.605400,10.174300,12.023800,3.248800,5.319500,9.006600
50%,54.000000,9.405400,7.087600,11.761400,10.870000,10.494600,12.275000,4.040900,6.522400,9.581200
75%,81.000000,9.918400,8.983200,12.054000,11.034400,10.732800,12.561900,4.614500,7.993400,10.052100
max,108.000000,11.079000,12.236500,12.892800,11.981700,12.236900,14.064300,6.595000,10.594400,11.611000


In [251]:
##normalization
min_max_scaler=MinMaxScaler()


In [252]:
x_train_norm=min_max_scaler.fit_transform(x_train)
x_test_norm=min_max_scaler.fit_transform(x_test)

In [253]:
x_train.iloc[0,3]

np.float64(12.0409)

In [254]:
x_train_norm[0,3]

np.float64(0.5572491159614685)

In [255]:
###feature selection
from sklearn.feature_selection import mutual_info_classif
mi=mutual_info_classif(x_train_norm,y_train) 

In [256]:
mi.shape

(46954,)

In [257]:
mi[0:10]

array([7.77156117e-16, 7.77156117e-16, 7.77156117e-16, 7.77156117e-16,
       7.77156117e-16, 7.77156117e-16, 7.77156117e-16, 7.77156117e-16,
       7.77156117e-16, 7.77156117e-16])

In [258]:
features=x_train.columns
features.shape

(46954,)

In [259]:
###selecting top500features
n_features=16379
selected_score_indices=np.argsort(mi)[::-1][0:n_features]

In [260]:
x_train_sel=x_train_norm[:,selected_score_indices]
x_test_sel=x_test_norm[:,selected_score_indices]

In [261]:
x_train_sel.shape, x_test_sel.shape

((87, 16379), (22, 16379))

In [262]:
### gene autoencoder


In [263]:
input_dim = 500  # Adjust to match your selected feature set
encoding_dim = 128  # Latent space size

# Redefine Autoencoder
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

input_layer = Input(shape=(input_dim,))
encoded = Dense(256, activation='relu')(input_layer)
encoded = Dense(encoding_dim, activation='relu')(encoded)

decoded = Dense(256, activation='relu')(encoded)
decoded = Dense(input_dim, activation='sigmoid')(decoded)

# Compile Model
gene_autoencoder = Model(inputs=input_layer, outputs=decoded)
gene_autoencoder.compile(optimizer="adam", loss="mse")

# Train Model
gene_autoencoder.fit(x_train_sel, x_train_sel, epochs=50, batch_size=32, validation_data=(x_test_sel, x_test_sel))


Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 470ms/step - loss: 0.0385 - val_loss: 0.0618
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 246ms/step - loss: 0.0343 - val_loss: 0.0628
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 225ms/step - loss: 0.0263 - val_loss: 0.0627
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 216ms/step - loss: 0.0277 - val_loss: 0.0602
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 246ms/step - loss: 0.0249 - val_loss: 0.0594
Epoch 6/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 230ms/step - loss: 0.0241 - val_loss: 0.0597
Epoch 7/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 218ms/step - loss: 0.0247 - val_loss: 0.0589
Epoch 8/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 234ms/step - loss: 0.0239 - val_loss: 0.0583
Epoch 9/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 220ms/step - loss: 0.0241 - val_loss: 0.0577
Epoch 10/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 241ms/step - loss: 0.0241 - val_loss: 0.0568
Epoch 11/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 240ms/step - loss: 0.0240 - val_loss: 0.0564
Epoch 12/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 229ms/step - loss: 0.0206 - val_lo

In [264]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D
from tensorflow.keras.models import Model

IMG_SIZE = (256, 256)
BATCH_SIZE = 32

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_data = datagen.flow_from_directory(
    directory="images/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='input',
    subset='training'
)

val_data = datagen.flow_from_directory(
    directory="images/valid",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='input',
    subset='validation'
)


Found 661 images belonging to 3 classes.
Found 72 images belonging to 3 classes.


In [265]:


# Autoencoder
input_img = Input(shape=(256, 256, 3))
x = Conv2D(32, (3, 3), activation='relu', padding='same')(input_img)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2), padding='same')(x)
encoded = Conv2D(128, (3, 3), activation='relu', padding='same')(x)

x = Conv2D(64, (3, 3), activation='relu', padding='same')(encoded)
x = UpSampling2D((2, 2))(x)
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)
decoded = Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)

img_autoencoder = Model(input_img, decoded)
img_autoencoder.compile(optimizer='adam', loss='mse')


In [266]:
img_autoencoder.summary()

Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)      │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (None, 256, 256, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 128, 128, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_20 (Conv2D)              │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_21 (Conv2D)              │ (None, 64, 64, 64)     │        73,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_6 (UpSampling2D)  │ (None, 128, 128, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_22 (Conv2D)              │ (None, 128, 128, 32)   │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_7 (UpSampling2D)  │ (None, 256, 256, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_23 (Conv2D)              │ (None, 256, 256, 3)    │           867 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 186,371 (728.01 KB)

 Trainable params: 186,371 (728.01 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train the imageautoencoder
img_autoencoder.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    steps_per_epoch=len(train_data),
    validation_steps=len(val_data),
    batch_size=BATCH_SIZE
)



C:\Users\prana\tensorflow_env\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
15/21 ━━━━━━━━━━━━━━━━━━━━ 17s 3s/step - loss: 0.0906  

In [ ]:
label_data_gen = ImageDataGenerator(rescale=1./255, validation_split=0.5)

train_label_data = label_data_gen.flow_from_directory(
    directory="images/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_label_data = label_data_gen.flow_from_directory(
    directory="images/valid",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

In [ ]:
# Create Encoder Model for feature extraction
encoder = Model(img_autoencoder.input, img_autoencoder.layers[5].output)

image_features_train = img_autoencoder.predict(train_data)
image_features_test = img_autoencoder.predict(val_data)


# Flatten the image features to match gene feature shape for fusion
image_features_train = image_features_train.reshape(image_features_train.shape[0], -1)
image_features_test = image_features_test.reshape(image_features_test.shape[0], -1)

print("Flattened Image Feature Shape;", image_features_train.shape)
print("flattened image test features;", image_features_test.shape)

# Save extracted features for fusion
np.save("train_image_features.npy", image_features_train)
np.save("val_image_features.npy", image_features_test)


In [ ]:

print(image_features_train)

In [ ]:
input_layer = Input(shape=(input_dim,))
encoded = Dense(256, activation='relu')(input_layer)
encoded = Dense(encoding_dim, activation='relu')(encoded)

decoded = Dense(256, activation='relu')(encoded)
decoded = Dense(input_dim, activation='sigmoid')(decoded)

gene_autoencoder = Model(inputs=input_layer, outputs=decoded)


In [ ]:
gene_encoder = Model(inputs=input_layer, outputs=encoded)


In [ ]:
gene_features_train = gene_encoder.predict(x_train_sel)
gene_features_test = gene_encoder.predict(x_test_sel)



In [ ]:
print("Gene features train shape:", x_train_sel.shape)
print("Image features train shape:", image_features_train.shape)
print("gfts", x_test_sel.shape)
print("imgfts", image_features_test.shape)

In [ ]:
##Classifier

In [ ]:
mdf= pd.read_csv("meregenew214.csv")
mdf.columns[0:10]
mdf.shape

In [ ]:
mdf = mdf.set_index('sample_id')
gene_data = mdf.set_index('sample_id.1')

In [ ]:
# 1. Drop non-numeric columns from gene_df (keep only gene expression)
ngd = gene_data.select_dtypes(include=[np.number])

In [ ]:
print(type(gene_data.index), gene_data.index[:5])
print(type(mdf.index), mdf.index[:5])

In [ ]:

meta_input =mdf.loc[mdf.index]

In [ ]:

# Filter your current gene data
gene_input = ngd.loc[mdf['sample_id.1']].values.astype('float32')


In [ ]:
gene_input.shape


In [ ]:
gene_autoencoder.input_shape

In [ ]:
gene_latent = gene_autoencoder.predict(gene_input)

In [ ]:
# Autoencoder
input_img = Input(shape=(256, 256, 3))
x = Conv2D(32, (3, 3), activation='relu', padding='same')(input_img)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2), padding='same')(x)
encoded = Conv2D(128, (3, 3), activation='relu', padding='same')(x)

x = Conv2D(64, (3, 3), activation='relu', padding='same')(encoded)
x = UpSampling2D((2, 2))(x)
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)
decoded = Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)

img_autoencoder = Model(input_img, decoded)


In [ ]:
print(type(meta_input))

In [ ]:
import numpy as np

# If your images are paths, load them like this:
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.image import resize

def preprocess_images(image_paths, target_size=(256, 256)):
    image_array = []
    for path in image_paths:
        img = load_img(path, target_size=target_size)
        img_array = img_to_array(img) / 255.0
        image_array.append(img_array)
    return np.array(image_array)

# Example:
image_paths = mdf['filename'].tolist()  # Assuming your metadata has image paths
image_input = preprocess_images(image_paths)


In [ ]:
image_input.shape

In [ ]:
image_latent = img_autoencoder.predict(image_input)

In [ ]:
image_latent_flat = image_latent.reshape((image_latent.shape[0], -1))

print(gene_latent.shape, image_latent_flat.shape)


In [ ]:
import numpy as np
fused_features = np.concatenate([gene_latent, image_latent_flat], axis=1)


In [ ]:
# Replace 'label' with your actual class/target column
labels = mdf['labels'].values


In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

X_train, X_test, Y_train, Y_test = train_test_split(fused_features, labels, test_size=0.2, random_state=42)



In [ ]:
X_train.dtype

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Initialize the label encoder
label_encoder = LabelEncoder()

# Fit and transform the labels to numeric values
Y_train = label_encoder.fit_transform(Y_train)

# Now, you can convert the labels to float32 if needed
Y_train = Y_train.astype('float32')


In [ ]:
classifier.fit(X_train, Y_train)

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(
    learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
model.fit(X_train,Y_train,epochs=50,validation_split=0.2,callbacks=[early_stopping])

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Initialize the label encoder
label_encoder = LabelEncoder()

# Fit and transform the labels to numeric values
Y_test = label_encoder.fit_transform(Y_test)

# Now, you can convert the labels to float32 if needed
Y_test = Y_test.astype('float32')


In [ ]:
 loss, acc = model.evaluate(X_test, Y_test)

In [ ]:
model.save('autoencoder.keras')

In [ ]:
labels

In [ ]:
model.summary()

In [ ]:
import numpy as np

# Example array of class labels (as provided)
y_labels = np.array([
    'benign train', 'benign train', 'malignant train', 'malignant train', 'malignant train',
    'normal', 'malignant train', 'benign train', 'malignant train', 'benign train', 'benign train',
    'malignant train', 'benign train', 'benign train', 'benign train', 'malignant train',
    'malignant train', 'malignant train', 'malignant train', 'malignant train', 'malignant train',
    'malignant train', 'malignant train', 'malignant train', 'benign train', 'benign train',
    'malignant train', 'benign train', 'benign train', 'benign train', 'malignant train',
    'benign train', 'benign train', 'malignant train', 'malignant train', 'malignant train',
    'malignant train', 'malignant train', 'malignant train', 'malignant train', 'malignant train',
    'malignant train', 'malignant train', 'malignant train', 'benign train', 'malignant train',
    'benign train', 'benign train', 'benign train', 'benign train', 'benign train', 'benign train',
    'benign train', 'benign train', 'benign train', 'malignant train', 'malignant train',
    'malignant train', 'malignant train', 'malignant train', 'benign train', 'malignant train',
    'benign train', 'benign valid', 'benign valid', 'malignant train', 'benign valid', 'benign valid',
    'benign valid', 'benign valid', 'benign valid', 'benign valid', 'malignant train', 'malignant train',
    'malignant train', 'malignant train', 'malignant train', 'benign valid', 'malignant train',
    'malignant train', 'malignant train', 'malignant train', 'malignant train', 'malignant train',
    'benign valid', 'benign valid', 'benign valid', 'benign valid', 'benign valid', 'malignant train',
    'malignant train', 'malignant train', 'malignant train', 'beningn', 'malignant train', 'malignant train',
    'malignant train', 'malignant train', 'beningn', 'beningn', 'benign valid', 'benign valid',
    'malignant train', 'malignant train', 'benign valid', 'malignant train', 'benign valid', 'malignant train',
    'benign valid', 'benign valid', 'benign valid'
])

# Clean up the labels
y_labels_cleaned = np.array([label.replace(" train", "").replace(" valid", "").replace("beningn", "benign") for label in y_labels])

# Check the cleaned labels
print(np.unique(y_labels_cleaned))  # This will give you the unique class names


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Initialize label encoder
label_encoder = LabelEncoder()

# Fit the label encoder and transform the cleaned labels
y_labels_encoded = label_encoder.fit_transform(y_labels_cleaned)

# Check the encoded labels and the mapping from labels to integers
print("Encoded Labels: ", y_labels_encoded)
print("Class Names Mapping: ", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Example: Assuming y_pred are the model predictions
# Here we'll assume you already have y_pred encoded as integers
# For demonstration, let's create a dummy set of predictions
y_pred = np.random.choice(np.unique(y_labels_encoded), size=len(y_labels_encoded))  # Simulate predictions

# Compute confusion matrix
cm = confusion_matrix(y_labels_encoded, y_pred)

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(
    learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])model.compile(optimizer=tf.keras.optimizers.Adam(
    learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])